
<a id='II-1'></a>
# <span style="color:#E74C3C">Part II — Time Series: Code Implementation</span>

We analyze **daily sales data** of a US Superstore from **2014 to 2017**, focusing on the **Office Supplies** category.

## **1. Import Libraries**

In [1]:
# ── Core data manipulation ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pylab import rcParams

# ── Statistical & Time Series Models ────────────────────────────────────────
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

# ── Auto ARIMA ───────────────────────────────────────────────────────────────
from pmdarima import auto_arima

# ── Metrics ──────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error

# ── Plot settings ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
sns.set_palette('husl')

print("All libraries imported successfully!")

All libraries imported successfully!


<a id='II-2'></a>
## **2. Load Dataset**

In [2]:
# Load dataset — update path as needed
df = pd.read_csv('data/raw/Dataset- Superstore (2014-2017).csv', encoding='latin-1')

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset Shape: (9994, 21)
Columns: ['ï»¿Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


,ï»¿Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print("=" * 50)
print("DATA TYPES & MISSING VALUES")
print("=" * 50)
info_df = pd.DataFrame({
    'Dtype':          df.dtypes,
    'Non-Null Count': df.notnull().sum(),
    'Null Count':     df.isnull().sum(),
    'Null %':         (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df)

print(f"\nTotal rows: {len(df):,}")
print(f"Date range: {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")

DATA TYPES & MISSING VALUES
                        Dtype  Non-Null Count  Null Count  Null %
ï»¿Row ID               int64            9994           0     0.0
Order ID               object            9994           0     0.0
Order Date     datetime64[ns]            9994           0     0.0
Ship Date      datetime64[ns]            9994           0     0.0
Ship Mode              object            9994           0     0.0
Customer ID            object            9994           0     0.0
Customer Name          object            9994           0     0.0
Segment                object            9994           0     0.0
Country                object            9994           0     0.0
City                   object            9994           0     0.0
State                  object            9994           0     0.0
Postal Code             int64            9994           0     0.0
Region                 object            9994           0     0.0
Product ID             object            9994   

In [4]:
# Category distribution
print("Category Distribution:")
print(df['Category'].value_counts())
print()
print("Sub-Category Distribution:")
print(df['Sub-Category'].value_counts())

Category Distribution:
Category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64

Sub-Category Distribution:
Sub-Category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64


<a id='II-3'></a>
## 3. **Data Preprocessing**

We focus on **Office Supplies** sales. Steps:
1. Filter to Office Supplies category
2. Parse dates correctly
3. Drop irrelevant columns
4. Aggregate sales by date

In [5]:
# Step 1: Parse date TRƯỚC khi filter để min/max hiển thị đúng
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Step 2: Filter to Office Supplies
OS = df.loc[df['Category'] == 'Office Supplies'].copy()

print(f"Office Supplies records: {len(OS):,}")
print(f"Date range: {OS['Order Date'].min()} → {OS['Order Date'].max()}")

# Step 4: Drop irrelevant columns
drop_cols = [
    'ï»¿Row ID', 'Order ID', 'Ship Date', 'Ship Mode',
    'Customer ID', 'Customer Name', 'Segment',
    'Country', 'City', 'State', 'Postal Code', 'Region',
    'Product ID', 'Category', 'Sub-Category', 'Product Name',
    'Quantity', 'Discount', 'Profit'
]
OS.drop(columns=drop_cols, inplace=True)

# Step 5: Check missing values
print(f"\nMissing values:\n{OS.isnull().sum()}")

# Step 6: Aggregate daily sales
OS_daily = OS.groupby('Order Date')['Sales'].sum().reset_index()
OS_daily.columns = ['Date', 'Sales']
print(f"\nUnique order dates (daily): {len(OS_daily)}")
OS_daily.head()

Office Supplies records: 6,026
Date range: 2014-01-03 00:00:00 → 2017-12-30 00:00:00

Missing values:
Order Date    0
Sales         0
dtype: int64

Unique order dates (daily): 1148


,Date,Sales
0,2014-01-03,16.448
1,2014-01-04,288.060
2,2014-01-05,19.536
3,2014-01-06,685.340
4,2014-01-07,10.430


<a id='II-4'></a>
## 4. **Time Series Indexing & Resampling**

In [6]:
# Set Date as index
OS_daily = OS_daily.set_index('Date')
OS_daily.index.freq = None  # Not all dates are present so no frequency

# Resample to Monthly (Mean of daily sales within each month)
# Using 'MS' (Month Start) for cleaner date labels
monthly_sales = OS_daily['Sales'].resample('MS').mean()
monthly_sales = monthly_sales.dropna()

print(f"Monthly data points: {len(monthly_sales)}")
print(f"From: {monthly_sales.index[0].strftime('%B %Y')}")
print(f"To:   {monthly_sales.index[-1].strftime('%B %Y')}")
print(f"\nDescriptive Statistics:")
print(monthly_sales.describe().round(2))

Monthly data points: 48
From: January 2014
To:   December 2017

Descriptive Statistics:
count      48.00
mean      600.16
std       289.94
min        63.04
25%       383.55
50%       550.05
75%       761.40
max      1357.06
Name: Sales, dtype: float64


In [7]:
import os
os.makedirs('outputs/processed', exist_ok=True)

import pickle
with open('outputs/processed/cleaned_data.pkl', 'wb') as f:
    pickle.dump({
        'df':            df,
        'OS_daily':      OS_daily,
        'monthly_sales': monthly_sales,
    }, f)

print("Saved!")


Saved!
